# Pipeline 2 bước đọc CCCD (bản Kaggle)
**Bước 1**: mô hình YOLOv8-seg tìm thẻ → lấy 4 góc → nắn thẳng bằng phép biến đổi phối cảnh (kích thước cố định 856x540).
**Bước 2**: mô hình YOLOv8 tìm vùng Số / Họ tên / Ngày sinh trên ảnh đã thẳng, nên khung ngang luôn khớp chữ.

**Trước khi chạy** (Session options): Accelerator = GPU T4 x2 hoặc P100, Internet = On.

**Cần 2 dataset trong Input** (để Private vì là dữ liệu nhạy cảm):
1. `dataset_yolo`: dataset hiện tại của bạn (Số / Họ tên / Ngày sinh).
2. `card_seg`: ảnh CCCD + nhãn **segmentation** 1 lớp `card`, mỗi thẻ là 1 đa giác 4 góc.
   - Dùng chính các ảnh đang có, chỉ cần chấm 4 góc thẻ (khoảng 100-200 ảnh, đủ các kiểu nghiêng / nền / ánh sáng).
   - Gán nhãn bằng Roboflow / CVAT / labelme (polygon), xuất định dạng **YOLOv8 segmentation**.
   - Cấu trúc: `data.yaml` (names: {0: card}), `images/train|val`, `labels/train|val`, mỗi dòng nhãn dạng `0 x1 y1 x2 y2 x3 y3 x4 y4` (toạ độ chuẩn hoá 0-1).

In [ ]:
!nvidia-smi
!pip install -q ultralytics

In [ ]:
import os, glob, shutil, zipfile, random
import yaml, cv2, numpy as np
from IPython.display import Image, display
from ultralytics import YOLO, settings
settings.update({"wandb": False})

INPUT = "/kaggle/input"
WORK = "/kaggle/temp"
OUT = "/kaggle/working"
CARD_ROOT = f"{WORK}/card_seg"
FIELD_ROOT = f"{WORK}/dataset_yolo"
RECT_ROOT = f"{WORK}/dataset_rect"
CARD_W, CARD_H = 856, 540          # tỉ lệ thẻ CCCD 85.6 x 54 mm

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)


def load_dataset(hint, dst):
    """Tìm dataset có đường dẫn chứa `hint` trong /kaggle/input (thư mục hoặc zip) rồi chép sang dst."""
    yamls = [p for p in glob.glob(f"{INPUT}/**/data.yaml", recursive=True) if hint in p]
    if yamls:
        shutil.copytree(os.path.dirname(yamls[0]), dst)
        return
    zips = [p for p in glob.glob(f"{INPUT}/**/*.zip", recursive=True) if hint in p]
    assert zips, f"Không thấy dataset '{hint}' trong /kaggle/input (tên thư mục hoặc zip phải chứa '{hint}')."
    tmp = f"{WORK}/_x_{hint}"
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(tmp)
    found = glob.glob(f"{tmp}/**/data.yaml", recursive=True)
    assert found, f"Không thấy data.yaml trong zip của '{hint}'"
    shutil.copytree(os.path.dirname(found[0]), dst)


def fix_yaml(root):
    p = f"{root}/data.yaml"
    cfg = yaml.safe_load(open(p, encoding="utf-8"))
    cfg["path"] = root
    cfg["train"] = "images/train"
    cfg["val"] = "images/val"
    cfg.pop("test", None)
    yaml.safe_dump(cfg, open(p, "w", encoding="utf-8"), allow_unicode=True)
    return cfg


def check_split(root):
    for sp in ["train", "val"]:
        imgs = [p for p in glob.glob(f"{root}/images/{sp}/*.*") if p.lower().endswith((".jpg", ".jpeg", ".png"))]
        miss = [os.path.basename(i) for i in imgs
                if not os.path.exists(f"{root}/labels/{sp}/{os.path.splitext(os.path.basename(i))[0]}.txt")]
        print(sp, "| ảnh:", len(imgs), "| ảnh thiếu nhãn:", miss[:5])
    for f in glob.glob(f"{root}/**/*.cache", recursive=True):
        os.remove(f)


def show(img, w=700):
    display(Image(data=cv2.imencode(".jpg", img)[1].tobytes(), width=w))

## A. Mô hình tìm thẻ (YOLOv8-seg)
Bật xoay + phối cảnh khi train vì đây chính là thứ mô hình này phải chịu được. Vẫn tắt `fliplr`.

In [ ]:
load_dataset("card_seg", CARD_ROOT)
print(fix_yaml(CARD_ROOT))
check_split(CARD_ROOT)

In [ ]:
card_model = YOLO("yolov8s-seg.pt")
card_model.train(
    data=f"{CARD_ROOT}/data.yaml",
    epochs=80, imgsz=640, batch=16, patience=20,
    fliplr=0.0, degrees=25, perspective=0.0005, translate=0.1, scale=0.4,
    device=0, project=f"{OUT}/runs", name="card", exist_ok=True,
)
CARD_BEST = f"{card_model.trainer.save_dir}/weights/best.pt"
print("card best.pt:", CARD_BEST)

## B. Hàm lấy 4 góc + nắn thẳng
- Lấy đa giác mask → bao lồi → xấp xỉ thành 4 góc (nếu không được thì dùng hình chữ nhật xoay nhỏ nhất).
- Sắp xếp góc theo chiều kim đồng hồ, cạnh dài nằm ngang, nên thẻ chụp dọc cũng được xoay về ngang.
- Còn lại chỉ có thể bị **ngược 180°**; phần suy luận ở mục E tự thử cả hai chiều.

In [ ]:
def card_quad(poly_xy):
    """Đa giác mask (N,2) -> 4 góc (4,2)."""
    cnt = np.asarray(poly_xy, dtype=np.float32).reshape(-1, 1, 2)
    hull = cv2.convexHull(cnt)
    peri = cv2.arcLength(hull, True)
    for eps in (0.02, 0.03, 0.05, 0.08):
        ap = cv2.approxPolyDP(hull, eps * peri, True)
        if len(ap) == 4:
            return ap.reshape(4, 2).astype(np.float32)
    return cv2.boxPoints(cv2.minAreaRect(hull)).astype(np.float32)


def order_quad(pts):
    """Trả về 4 góc theo thứ tự tl, tr, br, bl với cạnh tl->tr là cạnh dài."""
    pts = np.asarray(pts, np.float32)
    c = pts.mean(0)
    ang = np.arctan2(pts[:, 1] - c[1], pts[:, 0] - c[0])
    pts = pts[np.argsort(ang)]                 # chiều kim đồng hồ (trục y hướng xuống)
    pts = np.roll(pts, -int(np.argmin(pts.sum(1))), axis=0)
    w = np.linalg.norm(pts[1] - pts[0])
    h = np.linalg.norm(pts[2] - pts[1])
    if h > w:
        pts = np.roll(pts, -1, axis=0)
    return pts


def warp_card(img, quad):
    src = order_quad(quad)
    dst = np.float32([[0, 0], [CARD_W - 1, 0], [CARD_W - 1, CARD_H - 1], [0, CARD_H - 1]])
    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(img, M, (CARD_W, CARD_H), flags=cv2.INTER_CUBIC), M


def find_card(img, model, conf=0.25, imgsz=640):
    r = model.predict(img, imgsz=imgsz, conf=conf, verbose=False)[0]
    if r.masks is None or len(r.masks.xy) == 0:
        return None
    i = int(r.boxes.conf.argmax())
    poly = r.masks.xy[i]
    if len(poly) < 4:
        return None
    return card_quad(poly)

In [ ]:
# Kiểm tra bước 1 trên ảnh val của card_seg: ảnh ra phải là thẻ nằm ngang, thẳng, lấp đầy khung
card_best = YOLO(CARD_BEST)
for p in sorted(glob.glob(f"{CARD_ROOT}/images/val/*.*"))[:6]:
    img = cv2.imread(p)
    quad = find_card(img, card_best)
    if quad is None:
        print(os.path.basename(p), "-> KHÔNG tìm thấy thẻ")
        continue
    card, _ = warp_card(img, quad)
    show(card, 500)

## C. Tạo dataset đã nắn thẳng cho bước 2
Mỗi ảnh trong `dataset_yolo` được nắn thẳng bằng mô hình thẻ, nhãn cũ được biến đổi theo cùng phép phối cảnh.
- Nếu nhãn cũ là **polygon**, khung mới ôm theo polygon nên sẽ khít hơn.
- Nếu nhãn cũ là **hộp ngang** trên ảnh nghiêng, khung mới chỉ gần đúng và vẫn thừa/thiếu như cũ. Nhãn này chỉ để khởi đầu. Hãy tải `dataset_rect.zip` ở ô tiếp theo, sửa nhãn trên ảnh đã thẳng (lúc này chữ ngang nên rất nhanh), upload lại thành dataset mới rồi bỏ qua ô tạo dataset.
- Khi gán nhãn lại, kéo khung rộng ra ôm trọn cả chuỗi số (đủ 11 số, không cắt 2 đầu).

In [ ]:
def build_rect_dataset(card_model):
    cfg = yaml.safe_load(open(f"{FIELD_ROOT}/data.yaml", encoding="utf-8"))
    fallback = []
    for sp in ["train", "val"]:
        os.makedirs(f"{RECT_ROOT}/images/{sp}", exist_ok=True)
        os.makedirs(f"{RECT_ROOT}/labels/{sp}", exist_ok=True)
        paths = [p for p in sorted(glob.glob(f"{FIELD_ROOT}/images/{sp}/*.*"))
                 if p.lower().endswith((".jpg", ".jpeg", ".png"))]
        for p in paths:
            img = cv2.imread(p)
            if img is None:
                continue
            h, w = img.shape[:2]
            stem = os.path.splitext(os.path.basename(p))[0]
            quad = find_card(img, card_model)
            if quad is None:
                fallback.append(os.path.basename(p))
                card = cv2.resize(img, (CARD_W, CARD_H))
                M = np.array([[CARD_W / w, 0, 0], [0, CARD_H / h, 0], [0, 0, 1]], dtype=np.float64)
            else:
                card, M = warp_card(img, quad)
            cv2.imwrite(f"{RECT_ROOT}/images/{sp}/{stem}.jpg", card, [cv2.IMWRITE_JPEG_QUALITY, 95])

            out = []
            lp = f"{FIELD_ROOT}/labels/{sp}/{stem}.txt"
            if os.path.exists(lp):
                for line in open(lp):
                    v = line.split()
                    if len(v) < 5:
                        continue
                    c = int(float(v[0]))
                    nums = list(map(float, v[1:]))
                    if len(nums) == 4:
                        x, y, bw, bh = nums
                        pts = [[(x - bw / 2) * w, (y - bh / 2) * h], [(x + bw / 2) * w, (y - bh / 2) * h],
                               [(x + bw / 2) * w, (y + bh / 2) * h], [(x - bw / 2) * w, (y + bh / 2) * h]]
                    else:   # nhãn polygon
                        pts = [[nums[i] * w, nums[i + 1] * h] for i in range(0, len(nums) - 1, 2)]
                    t = cv2.perspectiveTransform(np.float32([pts]), M)[0]
                    x1, y1 = np.clip(t.min(0), 0, [CARD_W, CARD_H])
                    x2, y2 = np.clip(t.max(0), 0, [CARD_W, CARD_H])
                    if x2 - x1 < 2 or y2 - y1 < 2:
                        continue
                    out.append(f"{c} {(x1 + x2) / 2 / CARD_W:.6f} {(y1 + y2) / 2 / CARD_H:.6f} "
                               f"{(x2 - x1) / CARD_W:.6f} {(y2 - y1) / CARD_H:.6f}")
            open(f"{RECT_ROOT}/labels/{sp}/{stem}.txt", "w").write("\n".join(out))

    cfg["path"] = RECT_ROOT
    cfg["train"], cfg["val"] = "images/train", "images/val"
    cfg.pop("test", None)
    yaml.safe_dump(cfg, open(f"{RECT_ROOT}/data.yaml", "w", encoding="utf-8"), allow_unicode=True)
    print("Ảnh không tìm thấy thẻ (đã resize thẳng, cần xem lại):", fallback)
    return cfg


load_dataset("dataset_yolo", FIELD_ROOT)
fix_yaml(FIELD_ROOT)
check_split(FIELD_ROOT)
cfg = build_rect_dataset(card_best)
print(cfg)
check_split(RECT_ROOT)

In [ ]:
# Vẽ nhãn lên ảnh đã thẳng để kiểm tra khung có ôm trọn chuỗi chữ/số chưa
names = yaml.safe_load(open(f"{RECT_ROOT}/data.yaml", encoding="utf-8"))["names"]
names = list(names.values()) if isinstance(names, dict) else names
imgs = glob.glob(f"{RECT_ROOT}/images/train/*.jpg")
for p in random.sample(imgs, min(6, len(imgs))):
    im = cv2.imread(p)
    lb = f"{RECT_ROOT}/labels/train/{os.path.splitext(os.path.basename(p))[0]}.txt"
    for line in open(lb):
        if not line.strip():
            continue
        c, x, y, bw, bh = line.split()[:5]
        x, y, bw, bh = float(x) * CARD_W, float(y) * CARD_H, float(bw) * CARD_W, float(bh) * CARD_H
        p1, p2 = (int(x - bw / 2), int(y - bh / 2)), (int(x + bw / 2), int(y + bh / 2))
        cv2.rectangle(im, p1, p2, (0, 255, 0), 2)
        cv2.putText(im, names[int(c)], (p1[0], max(12, p1[1] - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    show(im, 700)

In [ ]:
# Đóng gói dataset đã nắn thẳng để tải về sửa nhãn (tab Output sau khi Save Version)
shutil.make_archive(f"{OUT}/dataset_rect", "zip", RECT_ROOT)
print("Đã tạo", f"{OUT}/dataset_rect.zip")

## D. Mô hình tìm Số / Họ tên / Ngày sinh trên ảnh đã thẳng
Ảnh đã thẳng nên **không cần** xoay / phối cảnh / lật khi augment (chỉ cho xoay rất nhẹ 3° để chịu sai số nắn thẳng). Ảnh 856x540 nên dùng `imgsz=960` để chữ không bị nhỏ.

In [ ]:
field_model = YOLO("yolov8s.pt")
field_model.train(
    data=f"{RECT_ROOT}/data.yaml",
    epochs=80, imgsz=960, batch=8, patience=20,
    fliplr=0.0, flipud=0.0, degrees=3, perspective=0.0, shear=0.0, translate=0.05, scale=0.2,
    device=0, project=f"{OUT}/runs", name="fields", exist_ok=True,
)
FIELD_BEST = f"{field_model.trainer.save_dir}/weights/best.pt"
print("field best.pt:", FIELD_BEST)

In [ ]:
field_best = YOLO(FIELD_BEST)
m = field_best.val(data=f"{RECT_ROOT}/data.yaml", imgsz=960, device=0)
print("mAP50:", round(m.box.map50, 3), "| mAP50-95:", round(m.box.map, 3))
print("Precision:", round(m.box.mp, 3), "| Recall:", round(m.box.mr, 3))
for name, ap in zip(m.names.values(), m.box.ap50):
    print(f"  {name}: AP50 = {ap:.3f}")

## E. Chạy cả pipeline trên ảnh gốc
Ảnh gốc → tìm thẻ → nắn thẳng → thử cả 2 chiều (0° và 180°), chọn chiều có tổng độ tin cậy cao hơn → tìm 3 trường → cắt ảnh từng trường.
`pad_x`, `pad_y` nới khung vài pixel khi cắt để chắc chắn không mất đầu/đuôi chuỗi (hữu ích khi đưa vào OCR).

In [ ]:
def read_card(img, card_model, field_model, conf=0.25, pad_x=10, pad_y=5):
    quad = find_card(img, card_model)
    if quad is None:
        card, found = cv2.resize(img, (CARD_W, CARD_H)), False
    else:
        card, _ = warp_card(img, quad)
        found = True

    best = None
    for cand in (card, cv2.rotate(card, cv2.ROTATE_180)):
        r = field_model.predict(cand, imgsz=960, conf=conf, verbose=False)[0]
        det = {}
        for b in r.boxes:
            name = field_model.names[int(b.cls)]
            c = float(b.conf)
            if name not in det or c > det[name][0]:
                det[name] = (c, b.xyxy[0].cpu().numpy())
        score = sum(c for c, _ in det.values())
        if best is None or score > best[0]:
            best = (score, cand, det)

    _, card, det = best
    crops = {}
    for name, (c, (x1, y1, x2, y2)) in det.items():
        x1, y1 = max(0, int(x1 - pad_x)), max(0, int(y1 - pad_y))
        x2, y2 = min(CARD_W, int(x2 + pad_x)), min(CARD_H, int(y2 + pad_y))
        crops[name] = card[y1:y2, x1:x2]
    return card, det, crops, found

In [ ]:
os.makedirs(f"{OUT}/demo_crops", exist_ok=True)
for p in sorted(glob.glob(f"{FIELD_ROOT}/images/val/*.jpg"))[:6]:
    img = cv2.imread(p)
    card, det, crops, found = read_card(img, card_best, field_best)
    vis = card.copy()
    for name, (c, (x1, y1, x2, y2)) in det.items():
        cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(vis, f"{name} {c:.2f}", (int(x1), max(12, int(y1) - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    print(os.path.basename(p), "| tìm thấy thẻ:", found, "| trường:", list(det))
    show(vis, 700)
    stem = os.path.splitext(os.path.basename(p))[0]
    for name, cr in crops.items():
        cv2.imwrite(f"{OUT}/demo_crops/{stem}_{name}.jpg", cr)

## F. Lưu mô hình
Sau khi **Save Version → Save & Run All (Commit)**, tải `card_best.pt` và `field_best.pt` ở tab **Output**. Khi dùng thật, nạp cả hai mô hình rồi gọi `read_card(img, card_model, field_model)` (copy các hàm ở mục B và E).

In [ ]:
shutil.copy(CARD_BEST, f"{OUT}/card_best.pt")
shutil.copy(FIELD_BEST, f"{OUT}/field_best.pt")
print("Đã lưu card_best.pt và field_best.pt vào", OUT)